# ProtGPT API — inference example
Load a trained checkpoint and run inference on the flashlfq proteomics train set.

Two entry points:
- `compute_sst(...)` → one whole-sample (SST) embedding per sample.
- `predict(...)` → predict held-out expression bins + metrics (the masked-modeling eval).

In [1]:
%load_ext autoreload
%autoreload 2

import os
# Run from the repo root so the config's relative fasta/ and esmc_cache paths resolve
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("cwd:", os.getcwd())

from protgpt.api import compute_sst, predict, visualize_sst
from protgpt.data import ExpressionDataset

CKPT = "model/trans_prot_mapped/best_model.ckpt"
DATA = "data/sc_trans/protein_mapped/train.h5ad"

cwd: c:\Users\sander\OneDrive\Bureaublad\Projects\prot_GPT\code\protgpt


## 1. `compute_sst` — whole-sample (SST) embeddings
Every detected protein is used as context, so each SST embedding summarizes the full proteome. Reads num_bins/detect_groups/ESM-C settings from the checkpoint config and rebuilds the ESM-C lookup for this dataset's proteins.

In [2]:
out = compute_sst(CKPT, DATA, device="cuda", batch_size=64, num_workers=0)
print("SST embeddings:", out["sst_emb"].shape)   # (n_samples, d_model)
print("n samples     :", len(out["sample_ids"]))
print("sample id [0]  :", out["sample_ids"][0])

Running model: 100%|██████████| 55413/55413 [48:18<00:00, 19.12it/s] 


SST embeddings: (3546382, 256)
n samples     : 3546382
sample id [0]  : cell_0


In [3]:
ds = ExpressionDataset(DATA, num_bins=10, detect_groups=True, max_group_size=1)
print("obs columns:", list(ds.obs.columns))
ds.obs.head(3)

Building cache: 100%|██████████| 433/433 [12:29<00:00,  1.73s/it]


obs columns: ['dataset_id', 'cell_type', 'tissue', 'tissue_general', 'disease', 'donor_id', 'sex', 'assay', 'development_stage', 'self_reported_ethnicity', 'raw_sum', 'nnz', 'is_primary_data', 'suspension_type']


,dataset_id,cell_type,tissue,tissue_general,disease,donor_id,sex,assay,development_stage,self_reported_ethnicity,raw_sum,nnz,is_primary_data,suspension_type
cell_id,,,,,,,,,,,,,,
cell_0,8b2e5453-faf7-46ea-9073-aea69b283cb7,basal cell of prostate epithelium,transition zone of prostate,prostate gland,benign prostatic hyperplasia,BPH342,male,10x 3' v2,57-year-old stage,European American,10368.0,2528,True,cell
cell_1,8b2e5453-faf7-46ea-9073-aea69b283cb7,prostate gland microvascular endothelial cell,transition zone of prostate,prostate gland,benign prostatic hyperplasia,BPH342,male,10x 3' v2,57-year-old stage,European American,6105.0,1848,True,cell
cell_2,8b2e5453-faf7-46ea-9073-aea69b283cb7,basal cell of prostate epithelium,transition zone of prostate,prostate gland,benign prostatic hyperplasia,BPH342,male,10x 3' v2,57-year-old stage,European American,6150.0,1308,True,cell


## 2. UMAP of the SST embeddings, coloured by cell type

3.5M points can't all be UMAP'd, and balancing across all ~614 cell types fills the plot
with rare and ambiguous labels (`unknown`, `stromal cell`, `malignant cell`, and the many
overlapping T-cell / epithelial / endothelial subtypes) that smear into each other. Instead
we focus on a **curated set of well-defined, lineage-distinct cell types**: sample up to
`CAP` cells of each from the already-computed embeddings and UMAP only those, so the major
lineages form clean, separable clusters. Edit `CURATED` to change the set.

In [15]:
# Curated, lineage-distinct cell types only — edit CURATED to taste. We sample up to CAP
# cells of each from the already-computed SST embeddings (no re-embedding) and UMAP those.
# Names must match the dataset's Cell-Ontology labels exactly; any that don't are reported
# below and skipped, so it's safe to over-include and prune from the printout.
import numpy as np

CURATED = [
    # lymphoid
    "T cell", "B cell", "natural killer cell", "plasma cell",
    # myeloid / granulocyte
    "monocyte", "macrophage", "dendritic cell", "mast cell", "neutrophil",
    # erythroid / megakaryocyte / progenitor
    "erythrocyte", "megakaryocyte", "hematopoietic stem cell",
    # stroma / mural / vascular / adipose
    "fibroblast", "smooth muscle cell", "pericyte", "endothelial cell",
    "preadipocyte", "adipocyte",
    # epithelium (distinct sub-lineages)
    "epithelial cell", "keratinocyte", "basal cell", "ciliated cell",
    "goblet cell", "hepatocyte", "enterocyte", "type II pneumocyte",
    # neural / glia / neural-crest
    "neuron", "astrocyte", "oligodendrocyte", "microglial cell", "melanocyte",
    # striated muscle
    "cardiac muscle cell", "skeletal muscle cell",
    # placenta (largest tissue in this atlas)
    "trophoblast cell",
]
CAP  = 2000     # cells per cell type (capped at availability)
SEED = 0

ct_all = ds.obs["cell_type"].astype(str).to_numpy()
present = [c for c in CURATED if (ct_all == c).sum() > 0]
missing = [c for c in CURATED if c not in present]
if missing:
    print(f"not in data (skipped): {missing}")

rng = np.random.default_rng(SEED)
picked = []
for c in present:
    pos = np.where(ct_all == c)[0]
    if pos.size > CAP:
        pos = rng.choice(pos, CAP, replace=False)
    picked.append(pos)
sel = np.sort(np.concatenate(picked))

out_sub = {"sst_emb": out["sst_emb"][sel],
           "sample_ids": [out["sample_ids"][i] for i in sel]}
sub_obs = ds.obs.iloc[sel].copy()
print(f"{len(sel):,} cells across {len(present)} curated cell types:")
print(sub_obs["cell_type"].value_counts())

not in data (skipped): ['type II pneumocyte', 'skeletal muscle cell', 'trophoblast cell']
59,107 cells across 31 curated cell types:
cell_type
smooth muscle cell               2000
plasma cell                      2000
preadipocyte                     2000
pericyte                         2000
neutrophil                       2000
                                 ... 
tonsil germinal center B cell       0
tongue muscle cell                  0
tissue-resident macrophage          0
tip cell                            0
thyroid follicular cell             0
Name: count, Length: 898, dtype: int64


In [25]:
# Original Dark24+Light24 palette, but with the legend SORTED BY COLOUR (hue) so the
# swatches form a smooth progression instead of a random scatter of similar tones.
# Each curated type keeps an original-palette colour; we just reorder the legend by hue.
import colorsys
import plotly.express as px

base = (px.colors.qualitative.Dark24 + px.colors.qualitative.Light24)[:len(present)]

def _hue_key(hexc):
    r, g, b = (int(hexc.lstrip("#")[i:i + 2], 16) / 255 for i in (0, 2, 4))
    h, s, v = colorsys.rgb_to_hsv(r, g, b)
    return (h, v)                       # sort by hue, then brightness

pairs   = sorted(zip(present, base), key=lambda cp: _hue_key(cp[1]))
order   = [c for c, _ in pairs]         # legend order
palette = [col for _, col in pairs]     # colours, hue-sorted to match

fig = visualize_sst(
    out_sub,
    metadata=sub_obs,
    color_by="cell_type",
    color_order=order,
    palette=palette,
    hover=["cell_type", "tissue_general", "assay", "disease"],
    width=1600,
    height=1200,
    method="umap",
    title=f"SST embeddings — {len(sel):,} cells, {sub_obs['cell_type'].nunique()} curated cell types (UMAP)",
    n_neighbors=100,
    min_dist=0.9,
    metric="cosine",
    random_state=SEED,
)

c:\Users\sander\miniconda3\envs\prot_gpt\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
